# Setup

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Suppress warnings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load

In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

print(model)

W1012 07:41:49.829000 16976 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1012 07:41:49.902000 16976 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [ ]:
from torchinfo import summary

x = tokenizer(
    "I love natural language processing.",
    return_tensors="pt"
)

x = {k: v.to(device) for k, v in x.items()}

model_summary = summary(
    model,
    input_data=x,
    depth=4,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    verbose=0
)

with open("../tmp/distilbert_summary.txt", "w", encoding="utf-8") as f:
    f.write(str(model_summary))

# 2. Embedding

- Word embedding
  - Fixed (V, d) lookup table, where V = vocab size and d = embedding dimension.
- Positional embedding
  - Absolute Positional Embedding (APE)
    - Another fixed (T, d) lookup table, where T = maximum number of tokens of the model.
    - Sinusoidal Positional Encoding: from original transformer paper, using sin/cos.
    - Learned Absolute Positional Embedding: used in early transformers, e.g. BERT or GPT-2.
  - Rotary Positional Embeddding (RoPE)
    - $Q' = R_m Q, \quad K' = R_m K$, where $m$ = token position.
    - $R_m =
\begin{bmatrix}
\cos(m\theta) & -\sin(m\theta) \\
\sin(m\theta) & \cos(m\theta)
\end{bmatrix}$, $\theta_i = 10000^{-2i/d_{\text{head}}}$, where $i$ = $0, 1, ..., (d_{head}/2 - 1$)

In [4]:
from transformers import AutoModel

model_name = 'distilbert-base-uncased'
model = AutoModel.from_pretrained(model_name)

embeddings = model.embeddings

## 2.1. Word Embedding

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer('I love you')['input_ids']

word_emb_layer = embeddings.word_embeddings
word_emb_matrix = next(word_emb_layer.parameters())
word_emb = word_emb_matrix[tokens]

print(word_emb_matrix.shape)    # (V, d), V=vocab_size, d=embedding dimension
print(word_emb.shape)           # (L, d), L=n_tokens

torch.Size([30522, 768])
torch.Size([5, 768])


## 2.2. Positional Embedding

### Learned APE

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer('I love you')['input_ids']

position_ids = embeddings.position_ids[0][:len(tokens)]     # [0, 1, 2, 3, 4]
position_emb_matrix = next(embeddings.position_embeddings.parameters())
position_emb = position_emb_matrix[position_ids]

print(position_emb_matrix.shape)    # (T, d), T=max_tokens
print(position_emb.shape)           # (L, d)

torch.Size([512, 768])
torch.Size([5, 768])


### RoPE

1. Compute rotation angles
  - $\theta_{p,j}=\frac{p}{\text{base}^{2j/d}}$, where $j=0,\ldots,d/2-1$.

2. Rotate Q and K
  - $\begin{bmatrix}q'_{p,j}\\q'_{p,j+d/2}\end{bmatrix}=\begin{bmatrix}\cos\theta_{p,j}&-\sin\theta_{p,j}\\\sin\theta_{p,j}&\cos\theta_{p,j}\end{bmatrix}\begin{bmatrix}q_{p,j}\\q_{p,j+d/2}\end{bmatrix}$

3. Compute attention scores
  - $S=\frac{Q'K'^\top}{\sqrt d}$

In [ ]:
from transformers import AutoModel
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb

model = AutoModel.from_pretrained("Qwen/Qwen2.5-0.5B")
rope_layer = model.rotary_emb

# Example query and key tensors
q = torch.randn(1, 14, 4, 64)  # [batch, query_heads, seq_len, head_dim]
k = torch.randn(1, 2, 4, 64)   # [batch, key_heads, seq_len, head_dim]

# Token positions
position_ids = torch.arange(4).unsqueeze(0)

# Generate rotary embeddings
cos, sin = rope_layer(q, position_ids)

# Apply RoPE
q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)

print(q_rot.shape)  # torch.Size([1, 14, 4, 64])
print(k_rot.shape)  # torch.Size([1, 2, 4, 64])

# 3. Normalization

- Each token $x = [x_1, ..., x_{768}]$ normalizes its 768 features.
- Pre-Norm vs Post-Norm
  - Pre-Norm
    - LayerNorm -> Attention
    - Dominant for modern transformers.
  - Post-Norm
    - Attention -> LayerNorm
    - Original transformers like BERT or DistilBERT.
- Normalization Methods
  - $\operatorname{LayerNorm}(x) =
  \gamma \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$
    - Typical standardization: $x_i' = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$
      - $\mu$ and $\sigma$ are calculated for each token individually, not accross batch or sequence.
    - Learned scale and shift: $y_i = \gamma_i x_i' + \beta_i$
  - $\operatorname{RMSNorm}(x) =
  \gamma \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}$
    - Commonly paired with pre-norm architectures in modern LLMs.
    - Slightly less computation, and empirically strong enough.

In [8]:
x = word_emb + position_emb         # (L, d)
x = model.embeddings.LayerNorm(x)   # (L, d)

# torch implementation
ln = nn.LayerNorm(768)
rms = nn.RMSNorm(768)

# 4. Attention

## 4.1. Self-Attention

In [9]:
x = word_emb + position_emb     # embedding: (L, d)

attn_layer = model.transformer.layer[0].attention
print(attn_layer)

wq = attn_layer.q_lin       # (d, d)
wk = attn_layer.k_lin       # (d, d)
wv = attn_layer.v_lin       # (d, d)

q = wq(x)      # query: (L, d), what information each token is looking for.
k = wk(x)      # key: (L, d), what features each token offers for matching with queries.
v = wv(x)      # value: (L, d), what information each token provides when attended to.

d = q.shape[1]
attn_score = q.matmul(k.T) / d**0.5     # (L, L): how much each token is related to others.

attn_weights = attn_score.softmax(dim=-1)   # (L, L)
attn_output = attn_weights @ v              # (L, d)

attn_output.shape

DistilBertSelfAttention(
  (q_lin): Linear(in_features=768, out_features=768, bias=True)
  (k_lin): Linear(in_features=768, out_features=768, bias=True)
  (v_lin): Linear(in_features=768, out_features=768, bias=True)
  (out_lin): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


torch.Size([5, 768])

## 4.2. Multi-Head Attention

In [10]:
attn_layer = model.transformer.layer[0].attention

h = model.config.n_heads  # h = 12
d = model.config.dim      # d = 768
d_head = d // h           # d_head = 64

q = attn_layer.q_lin(x)   # (L, 768)
k = attn_layer.k_lin(x)   # (L, 768)
v = attn_layer.v_lin(x)   # (L, 768)

# Split into attention heads
q = q.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)
k = k.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)
v = v.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)

attn_score = q @ k.transpose(-2, -1) / d_head**0.5  # (12, L, L)
attn_weights = attn_score.softmax(dim=-1)            # (12, L, L)
attn_output = attn_weights @ v                        # (12, L, 64)

## 4.3. GQA, Grouped-Query Attention

- Calculates $QK^T$.
- 1~7 Q share 1st K/V, 8~14 Q share 2nd K/V.

In [17]:
model_qwen = AutoModel.from_pretrained('Qwen/Qwen2.5-0.5B')
qwen_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')

attn_layer = model_qwen.layers[0].self_attn

print(attn_layer.q_proj)
print(attn_layer.k_proj)
print(attn_layer.v_proj)


input_ids = qwen_tokenizer(
    "I love you", return_tensors="pt"
)["input_ids"].to(model_qwen.device)

x_ = model_qwen.embed_tokens(input_ids)
q = attn_layer.q_proj(x_)
k = attn_layer.k_proj(x_)
v = attn_layer.v_proj(x_)

# Head sizes
n_q_heads = 14
n_kv_heads = 2
dim_head = 64

# Reshape
seq_len = x_.shape[1]
q = q.view(1, seq_len, n_q_heads, dim_head)
k = k.view(1, seq_len, n_kv_heads, dim_head)
v = v.view(1, seq_len, n_kv_heads, dim_head)

# Transpose: (B, L, H, D) -> (B, H, L, D)
q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)

print("Q: ", q.shape)
print("K: ", k.shape)
print("V: ", v.shape)

Linear(in_features=896, out_features=896, bias=True)
Linear(in_features=896, out_features=128, bias=True)
Linear(in_features=896, out_features=128, bias=True)
Q:  torch.Size([1, 14, 3, 64])
K:  torch.Size([1, 2, 3, 64])
V:  torch.Size([1, 2, 3, 64])


# 5. FFN

## 5.1. Dense

In [18]:
model = AutoModel.from_pretrained(model_name)

ffn_layer = model.transformer.layer[0].ffn

print(ffn_layer)

FFN(
  (dropout): Dropout(p=0.1, inplace=False)
  (lin1): Linear(in_features=768, out_features=3072, bias=True)
  (lin2): Linear(in_features=3072, out_features=768, bias=True)
  (activation): GELUActivation()
)


## 5.2. SwiGLU MLP

```markdown
x
├─→ Linear 1 → SiLU → gate
└─→ Linear 2        → values

gate * values
      ↓
   Linear 3
      ↓
    output

- gate_proj = Linear 1 = controls how much of each feature passes through.
- up_proj   = Linear 2 = creates features.
- down_proj = Linear 3 = filtered featrues.
```

In [19]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B')
mlp = model.model.layers[0].mlp

print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# 6. Causal mask

- Before softmax, mask future tokens.
  - $\begin{bmatrix}
0 & -\infty & -\infty \\
0 & 0 & -\infty \\
0 & 0 & 0
\end{bmatrix}$
- Automatically handled in `hf`.

In [20]:
q = torch.randn(2, 8, 10, 64)  # (batch, heads, seq_len, head_dim)
k = torch.randn(2, 8, 10, 64)
v = torch.randn(2, 8, 10, 64)

# With causal mask
output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

# Without causal mask
output = F.scaled_dot_product_attention(q, k, v, is_causal=False)

# 7. LM head

- Each token's embedding -> one score per vocabulary token.
- (B, L, 896) -> LM head -> (B, L, 151,936)

In [21]:
from transformers import AutoModelForCausalLM

model_clm = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B')

lm_head = model_clm.lm_head

print(model_clm)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# 8. KV cache

```markdown
Step 1:
"I"
K_I, V_I → cache

Step 2:
"love"
compute Q_love, K_love, V_love
attention uses:
Q_love vs [K_I, K_love]

cache becomes:
[K_I, K_love]
[V_I, V_love]
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_clm = AutoModelForCausalLM.from_pretrained(model_name)

# Inputs
inputs = tokenizer("Hello, my name is", return_tensors="pt")

# With KV Cache
outputs = model_clm.generate(**inputs, max_new_tokens=50, use_cache=True)

# Without KV Cache
outputs = model_clm.generate(**inputs, max_new_tokens=50, use_cache=False)